# Construção da One Big Table

A One Big Table (OBT) reúne em uma linha por transação os dados necessários à análise de fraude. Este notebook consolida transações e terminais, calcula atributos temporais e históricos, verifica o resultado e salva os conjuntos em Parquet. O treinamento do modelo será feito posteriormente.

Execute as células em ordem, desde o início. As consultas intermediárias dependem das etapas anteriores; para reconstruir tudo, reinicie o kernel e siga essa ordem. As células de exportação não sobrescrevem arquivos existentes.

## Instalação e imports das bibliotecas

O DuckDB executa as junções e os cálculos diretamente no banco, evitando carregar toda a base em DataFrames. Os DataFrames são usados aqui para exibir amostras e verificações.


In [28]:
!pip install pandas
!pip install numpy
!pip install duckdb pandas
!pip install matplotlib
!pip install seaborn


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import duckdb
from pathlib import Path
import matplotlib.pyplot as plt

## Conexão com o banco

O caminho abaixo considera a execução a partir de `program/notes`. Importamos os caminhos de `src/config.py`, que cria as pastas de dados automaticamente. O banco é procurado em `data/raw/tuca.db`; se não existir, usamos a cópia legada em `documents/tuca.db`. A conexão usa `read_only=True` para preservar `tuca.db`. Em seguida, conferimos as tabelas disponíveis e seus tipos de dados.


In [30]:
import sys

project_root = Path("../..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import ROOT, RAW_DIR, PROCESSED_DIR

path = RAW_DIR / "tuca.db"
if not path.exists():
    path = ROOT / "documents" / "tuca.db"

In [31]:
db = duckdb.connect(str(path), read_only=True)

tables = db.execute("SHOW TABLES").fetchall()
print(tables)

[('payers',), ('seller_terminals',), ('transactions',)]


In [32]:
for tabela in ["transactions", "payers", "seller_terminals"]:
    print(f"\nTabela: {tabela}")
    display(db.execute(f"DESCRIBE {tabela}").df())


Tabela: transactions


,column_name,column_type,null,key,default,extra
0,transaction_id,BIGINT,YES,None,None,None
1,tx_datetime,TIMESTAMP_NS,YES,None,None,None
2,tx_date,TIMESTAMP_NS,YES,None,None,None
3,tx_time,VARCHAR,YES,None,None,None
4,tx_amount,FLOAT,YES,None,None,None
5,card_id,BIGINT,YES,None,None,None
6,terminal_id,BIGINT,YES,None,None,None
7,is_transactional_fraud,BIGINT,YES,None,None,None
8,is_fraud,BOOLEAN,YES,None,None,None
9,tx_fraud_report_date,TIMESTAMP_NS,YES,None,None,None



Tabela: payers


,column_name,column_type,null,key,default,extra
0,card_hash,BIGINT,YES,None,None,None
1,card_bin,BIGINT,YES,None,None,None
2,card_first_transaction,VARCHAR,YES,None,None,None



Tabela: seller_terminals


,column_name,column_type,null,key,default,extra
0,terminal_id,BIGINT,YES,None,None,None
1,latitude,DOUBLE,YES,None,None,None
2,longitude,DOUBLE,YES,None,None,None
3,terminal_operation_start,VARCHAR,YES,None,None,None
4,terminal_soft_descriptor,VARCHAR,YES,None,None,None


## Consolidação das transações e dos terminais

`transactions` é a base da OBT. O `LEFT JOIN` associa o terminal pela chave `terminal_id`, preservando as transações mesmo quando não há correspondência. Cada terminal deve ter uma chave única em `seller_terminals`.

Não juntamos `payers` nesta versão porque os atributos selecionados não utilizam suas colunas. A primeira consulta inclui a data de início de operação do terminal, necessária para calcular sua idade.

A conferência seguinte deve apresentar o mesmo total na tabela original, na junção e na contagem de identificadores únicos.


In [33]:
query_obt = """
SELECT
    t.transaction_id,
    t.tx_datetime,
    t.tx_amount,
    t.card_id,
    t.terminal_id,
    s.terminal_operation_start,
    t.is_fraud,
    TRIM(
        REGEXP_REPLACE(s.terminal_soft_descriptor, '[0-9]+$', '')
    ) AS terminal_category
FROM transactions AS t
LEFT JOIN seller_terminals AS s
    ON t.terminal_id = s.terminal_id
"""

In [34]:
display(db.execute(query_obt + "\nLIMIT 10").df())

,transaction_id,tx_datetime,tx_amount,card_id,terminal_id,terminal_operation_start,is_fraud,terminal_category
0,1,2018-01-01 00:00:10,117.120003,4983922282101,1113,2017-11-12,False,Ifood BBQ Hub
1,2,2018-01-01 00:00:38,57.700001,38435765939628,790,2017-08-06,False,Modern Sofa Store
2,3,2018-01-01 00:00:38,41.500000,4053973451359964262,157,2017-01-09,False,Elite Hair Salon
3,4,2018-01-01 00:00:41,47.720001,370458387500772,850,2017-08-27,False,Cyber World
4,5,2018-01-01 00:01:46,14.530000,180022431672084,32,2016-08-29,False,Live Music Hub
5,6,2018-01-01 00:02:04,18.920000,4659202461752022,701,2017-07-13,False,Notepad Pro
6,7,2018-01-01 00:02:20,23.520000,2278929093043769,1179,2017-12-22,False,Rappi Pizza Express
7,8,2018-01-01 00:02:40,48.189999,30010034344997,206,2017-02-09,False,Local Market
8,9,2018-01-01 00:02:50,68.989998,30185438115076,630,2017-06-27,False,Taxi John
9,10,2018-01-01 00:02:57,16.100000,3543649677315964,460,2017-05-16,False,Island Escape


In [35]:
display(db.execute(f"""
    SELECT
        (SELECT COUNT(*) FROM transactions) AS total_original,
        COUNT(*) AS total_obt,
        COUNT(DISTINCT transaction_id) AS ids_unicos
    FROM ({query_obt}) AS obt
""").df())

,total_original,total_obt,ids_unicos
0,4483956,4483956,4483956


# Construção dos atributos

## Hora, dia da semana e idade do terminal

- `hour`: hora da transação, de 0 a 23.
- `day_week`: dia da semana, de 0 (domingo) a 6 (sábado).
- `terminal_age_days`: dias completos desde o início das operações até a transação.

A data de início do terminal está armazenada como texto, por isso usamos `TRY_CAST`. Datas inválidas, ausentes ou posteriores à transação produzem idade ausente. A conferência verifica os intervalos e a quantidade de idades ausentes.


In [36]:
query_obt = """
SELECT
    t.transaction_id,
    t.tx_datetime,
    t.tx_amount,
    t.card_id,
    t.terminal_id,
    TRIM(
        REGEXP_REPLACE(s.terminal_soft_descriptor, '[0-9]+$', '')
    ) AS terminal_category,

    EXTRACT(HOUR FROM t.tx_datetime) AS hour,
    EXTRACT(DOW FROM t.tx_datetime) AS day_week,

CASE
    WHEN CAST(t.tx_datetime AS TIMESTAMP)
         >= TRY_CAST(s.terminal_operation_start AS TIMESTAMP)
    THEN FLOOR(
        EXTRACT(
            EPOCH FROM (
                CAST(t.tx_datetime AS TIMESTAMP)
                - TRY_CAST(s.terminal_operation_start AS TIMESTAMP)
            )
        ) / 86400
    )
    ELSE NULL
END AS terminal_age_days,

    t.is_fraud

FROM transactions AS t
LEFT JOIN seller_terminals AS s
    ON t.terminal_id = s.terminal_id
"""

In [37]:
display(db.execute(query_obt + "\nLIMIT 10").df())

,transaction_id,tx_datetime,tx_amount,card_id,terminal_id,terminal_category,hour,day_week,terminal_age_days,is_fraud
0,704513,2018-01-29 14:09:02,12.220000,4544788937707052,1086,Airport Shuttle,14,1,87.0,False
1,704514,2018-01-29 14:09:03,329.519989,349770744347822,400,Tech Hub,14,1,275.0,False
2,704515,2018-01-29 14:09:03,28.790001,4466216785557605,1312,Quick Fix Plumbers,14,1,3.0,False
3,704516,2018-01-29 14:09:03,9.430000,180038570943912,331,VR Zone,14,1,301.0,False
4,704517,2018-01-29 14:09:05,76.709999,3524728639420121,454,Notepad Pro,14,1,259.0,False
5,704518,2018-01-29 14:09:05,14.590000,3523324165185485,1231,Quick Fix Plumbers,14,1,15.0,False
6,704519,2018-01-29 14:09:08,112.559998,4502743939192,1178,Prime Interiors,14,1,38.0,False
7,704520,2018-01-29 14:09:10,176.990005,569098802535,547,FastNet Internet,14,1,237.0,False
8,704521,2018-01-29 14:09:11,8.840000,4690973846896,105,Gold Star Car Wash,14,1,411.0,False
9,704522,2018-01-29 14:09:11,57.840000,4315509023075049,1175,Global Telecom,14,1,47.0,False


In [38]:
display(db.execute(f"""
    SELECT
        MIN(hour) AS menor_hora,
        MAX(hour) AS maior_hora,
        MIN(day_week) AS menor_dia,
        MAX(day_week) AS maior_dia,
        MIN(terminal_age_days) AS menor_idade,
        COUNT(*) FILTER (
            WHERE terminal_age_days IS NULL
        ) AS idades_ausentes
    FROM ({query_obt}) AS obt
""").df())

,menor_hora,maior_hora,menor_dia,maior_dia,menor_idade,idades_ausentes
0,0,23,0,6,0.0,0


## Terminal novo para o cartão

`terminal_new_for_card` vale 1 quando não existe uso anterior do mesmo terminal pelo cartão no histórico disponível, e 0 nos demais casos. Não significa que o equipamento acabou de iniciar suas operações.

Na primeira ocorrência do par cartão–terminal, o indicador vale 1. Transações no mesmo primeiro instante também recebem 1, pois não existe uso estritamente anterior. A amostra mostra pares com mais de uma transação para conferir a mudança do indicador.


In [39]:
query_obt_base = query_obt

In [40]:
query_obt = f"""
WITH base AS (
    {query_obt_base}
)
SELECT
    *,
    CASE
        WHEN tx_datetime = MIN(tx_datetime) OVER (
            PARTITION BY card_id, terminal_id
        )
        THEN 1
        ELSE 0
    END AS terminal_new_for_card
FROM base
"""

In [41]:
display(db.execute(f"""
    WITH obt AS (
        {query_obt}
    )
    SELECT
        card_id,
        terminal_id,
        transaction_id,
        tx_datetime,
        terminal_new_for_card
    FROM obt
    QUALIFY COUNT(*) OVER (
        PARTITION BY card_id, terminal_id
    ) > 1
    ORDER BY card_id, terminal_id, tx_datetime
    LIMIT 20
""").df())

,card_id,terminal_id,transaction_id,tx_datetime,terminal_new_for_card
0,60401818053,107,232431,2018-01-10 16:31:37,1
1,60401818053,107,3085454,2018-04-20 12:42:21,0
2,60401818053,107,3821747,2018-05-12 15:00:04,0
3,60401818053,163,1883181,2018-03-12 23:55:51,1
4,60401818053,163,3347261,2018-04-28 12:21:39,0
5,60401818053,205,917928,2018-02-06 15:59:34,1
6,60401818053,205,1042885,2018-02-11 11:34:44,0
7,60401818053,216,28578,2018-01-02 08:04:49,1
8,60401818053,216,1115511,2018-02-14 06:11:13,0
9,60401818053,216,2287150,2018-03-26 09:49:38,0


## Frequência de compras e diversidade de terminais

Calculamos quatro atributos do histórico do cartão e quatro do terminal:

- `card_tx_count_1h`: quantidade de transações anteriores em 1 hora.
- `card_tx_count_24h`: quantidade de transações anteriores em 24 horas.
- `card_distinct_terminals_24h`: quantidade de terminais diferentes usados nas 24 horas anteriores.
- `seconds_since_last_card_tx`: segundos desde a última transação estritamente anterior; ausente quando não há histórico.

As contagens do cartão representam atividade imediata e diária. A diversidade distingue várias compras no mesmo terminal de compras espalhadas por diferentes terminais. Para o terminal, as janelas medem a velocidade de uso e quantos cartões distintos passaram por ele; isso representa sinais de fraude de terminal e clonagem em vários cartões.

- `terminal_tx_count_1h` e `terminal_tx_count_24h`: transações anteriores no terminal.
- `terminal_distinct_cards_24h`: cartões distintos que usaram o terminal nas 24h anteriores.
- `seconds_since_last_terminal_tx`: intervalo desde o uso anterior do terminal.

As janelas usam `[t - janela, t)`. `EXCLUDE GROUP` exclui a transação atual e todas as transações com o mesmo instante. Sem histórico, as contagens valem 0.

Na conferência, as contagens de 1h não podem superar as de 24h, a diversidade não pode superar as transações da janela e os intervalos não ausentes devem ser positivos.


In [42]:
query_obt_terminal = query_obt

In [43]:
query_obt = f"""
WITH base AS (
    {query_obt_terminal}
)
SELECT
    *,
    COUNT(*) OVER (
        PARTITION BY card_id
        ORDER BY tx_datetime
        RANGE BETWEEN INTERVAL '1 hour' PRECEDING AND CURRENT ROW
        EXCLUDE GROUP
    ) AS card_tx_count_1h,

    COUNT(*) OVER (
        PARTITION BY card_id
        ORDER BY tx_datetime
        RANGE BETWEEN INTERVAL '24 hours' PRECEDING AND CURRENT ROW
        EXCLUDE GROUP
    ) AS card_tx_count_24h,

    COUNT(DISTINCT terminal_id) OVER (
        PARTITION BY card_id
        ORDER BY tx_datetime
        RANGE BETWEEN INTERVAL '24 hours' PRECEDING AND CURRENT ROW
        EXCLUDE GROUP
    ) AS card_distinct_terminals_24h,

    EXTRACT(
        EPOCH FROM (
            CAST(tx_datetime AS TIMESTAMP)
            - CAST(
                MAX(tx_datetime) OVER (
                    PARTITION BY card_id
                    ORDER BY tx_datetime
                    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                    EXCLUDE GROUP
                ) AS TIMESTAMP
            )
        )
    ) AS seconds_since_last_card_tx,

    COUNT(*) OVER (
        PARTITION BY terminal_id
        ORDER BY tx_datetime
        RANGE BETWEEN INTERVAL '1 hour' PRECEDING AND CURRENT ROW
        EXCLUDE GROUP
    ) AS terminal_tx_count_1h,

    COUNT(*) OVER (
        PARTITION BY terminal_id
        ORDER BY tx_datetime
        RANGE BETWEEN INTERVAL '24 hours' PRECEDING AND CURRENT ROW
        EXCLUDE GROUP
    ) AS terminal_tx_count_24h,

    COUNT(DISTINCT card_id) OVER (
        PARTITION BY terminal_id
        ORDER BY tx_datetime
        RANGE BETWEEN INTERVAL '24 hours' PRECEDING AND CURRENT ROW
        EXCLUDE GROUP
    ) AS terminal_distinct_cards_24h,

    EXTRACT(
        EPOCH FROM (
            CAST(tx_datetime AS TIMESTAMP)
            - CAST(
                MAX(tx_datetime) OVER (
                    PARTITION BY terminal_id
                    ORDER BY tx_datetime
                    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                    EXCLUDE GROUP
                ) AS TIMESTAMP
            )
        )
    ) AS seconds_since_last_terminal_tx
FROM base
"""

In [44]:
display(db.execute(f"""
    SELECT
        card_id,
        tx_datetime,
        card_tx_count_1h,
        card_tx_count_24h,
        card_distinct_terminals_24h,
        seconds_since_last_card_tx
    FROM ({query_obt}) AS obt
    ORDER BY card_id, tx_datetime
    LIMIT 20
""").df())

,card_id,tx_datetime,card_tx_count_1h,card_tx_count_24h,card_distinct_terminals_24h,seconds_since_last_card_tx
0,60401818053,2018-01-02 08:04:49,0,0,0,NaN
1,60401818053,2018-01-03 13:02:16,0,0,0,104247.0
2,60401818053,2018-01-05 11:55:02,0,0,0,168766.0
3,60401818053,2018-01-05 12:00:00,1,1,1,298.0
4,60401818053,2018-01-09 06:46:43,0,0,0,326803.0
5,60401818053,2018-01-10 11:57:57,0,0,0,105074.0
6,60401818053,2018-01-10 16:31:37,0,1,1,16420.0
7,60401818053,2018-01-11 16:58:18,0,0,0,88001.0
8,60401818053,2018-01-12 12:14:43,0,1,1,69385.0
9,60401818053,2018-01-14 21:31:26,0,0,0,206203.0


In [45]:
display(db.execute(f"""
    SELECT COUNT(*) AS linhas_inconsistentes
    FROM ({query_obt}) AS obt
    WHERE card_tx_count_1h > card_tx_count_24h
       OR card_distinct_terminals_24h < 0
       OR card_distinct_terminals_24h > card_tx_count_24h
       OR seconds_since_last_card_tx <= 0
       OR terminal_tx_count_1h > terminal_tx_count_24h
       OR terminal_distinct_cards_24h > terminal_tx_count_24h
       OR seconds_since_last_terminal_tx <= 0
""").df())

,linhas_inconsistentes
0,0


## Histórico de gasto em 30 dias

A janela de 30 dias foi escolhida como referência inicial para o padrão recente do cartão.

- `card_history_count_30d`: quantidade de valores não ausentes usados nas estatísticas.
- `card_amount_mean_30d`: média dos valores anteriores.
- `card_amount_std_30d`: desvio padrão amostral dos valores anteriores.

Sem valores anteriores, a média fica ausente; com menos de duas observações, o desvio também fica ausente. Esses casos são esperados e não devem ser descartados automaticamente.

O histórico inclui compras legítimas e fraudulentas sem filtrar por `is_fraud`, pois uma fraude ainda pode ser desconhecida naquele instante. Nenhum cálculo usa eventos futuros.


In [46]:
query_obt_counts = query_obt

In [47]:
query_obt = f"""
WITH base AS (
    {query_obt_counts}
)
SELECT
    *,
    COUNT(tx_amount) OVER history_30d AS card_history_count_30d,
    AVG(tx_amount) OVER history_30d AS card_amount_mean_30d,
    STDDEV_SAMP(tx_amount) OVER history_30d AS card_amount_std_30d
FROM base
WINDOW history_30d AS (
    PARTITION BY card_id
    ORDER BY tx_datetime
    RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
    EXCLUDE GROUP
)
"""

In [48]:
display(db.execute(f"""
    SELECT
        card_id,
        tx_datetime,
        tx_amount,
        card_history_count_30d,
        card_amount_mean_30d,
        card_amount_std_30d
    FROM ({query_obt}) AS obt
    ORDER BY card_id, tx_datetime
    LIMIT 20
""").df())

,card_id,tx_datetime,tx_amount,card_history_count_30d,card_amount_mean_30d,card_amount_std_30d
0,60401818053,2018-01-02 08:04:49,30.750000,0,NaN,NaN
1,60401818053,2018-01-03 13:02:16,53.360001,1,30.750000,NaN
2,60401818053,2018-01-05 11:55:02,110.099998,2,42.055000,15.987685
3,60401818053,2018-01-05 12:00:00,61.090000,3,64.736666,40.880031
4,60401818053,2018-01-09 06:46:43,78.050003,4,63.825000,33.428169
5,60401818053,2018-01-10 11:57:57,83.930000,5,66.670000,29.640378
6,60401818053,2018-01-10 16:31:37,42.139999,6,69.546667,27.431604
7,60401818053,2018-01-11 16:58:18,330.850006,7,65.631429,27.099466
8,60401818053,2018-01-12 12:14:43,44.040001,8,98.783751,97.067405
9,60401818053,2018-01-14 21:31:26,61.009998,9,92.701112,92.613754


## Z-score do valor da transação

O Z-score mede a distância do valor atual em relação ao padrão anterior do cartão:

`amount_z_score_30d = (tx_amount - card_amount_mean_30d) / card_amount_std_30d`

Valores positivos indicam compras acima da média; negativos, abaixo dela. O resultado permanece ausente quando o desvio é zero ou não está disponível. O Z-score é um atributo para o modelo, não uma decisão automática de fraude.


In [49]:
query_obt_stats = query_obt

In [50]:
query_obt = f"""
WITH base_estatisticas AS (
    {query_obt_stats}
),
base AS (
    SELECT
        *,
        CASE
            WHEN card_amount_std_30d > 0
            THEN (tx_amount - card_amount_mean_30d) / card_amount_std_30d
            ELSE NULL
        END AS amount_z_score_30d
    FROM base_estatisticas
),
historico_contagens AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY card_id
            ORDER BY tx_datetime, transaction_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS card_previous_tx_count,
        COUNT(DISTINCT terminal_id) OVER (
            PARTITION BY card_id
            ORDER BY tx_datetime, transaction_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS card_distinct_terminals,
        COUNT(*) OVER (
            PARTITION BY terminal_id
            ORDER BY tx_datetime, transaction_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS terminal_previous_tx_count
    FROM base
),
eventos_reportados AS (
    SELECT 'card' AS entity_type, CAST(card_id AS VARCHAR) AS entity_id,
        GREATEST(CAST(tx_datetime AS TIMESTAMP), CAST(tx_fraud_report_date AS TIMESTAMP)) AS event_time,
        1 AS event_type, NULL::BIGINT AS transaction_id, 1 AS reported_fraud
    FROM transactions WHERE tx_fraud_report_date IS NOT NULL
    UNION ALL
    SELECT 'terminal', CAST(terminal_id AS VARCHAR),
        GREATEST(CAST(tx_datetime AS TIMESTAMP), CAST(tx_fraud_report_date AS TIMESTAMP)),
        1, NULL::BIGINT, 1
    FROM transactions WHERE tx_fraud_report_date IS NOT NULL
),
eventos_transacao AS (
    SELECT 'card' AS entity_type, CAST(card_id AS VARCHAR) AS entity_id,
        CAST(tx_datetime AS TIMESTAMP) AS event_time, 0 AS event_type,
        transaction_id, 0 AS reported_fraud
    FROM historico_contagens
    UNION ALL
    SELECT 'terminal', CAST(terminal_id AS VARCHAR), CAST(tx_datetime AS TIMESTAMP),
        0, transaction_id, 0
    FROM historico_contagens
),
linha_do_tempo AS (
    SELECT * FROM eventos_transacao
    UNION ALL
    SELECT * FROM eventos_reportados
),
reportes_conhecidos AS (
    SELECT *, COALESCE(SUM(reported_fraud) OVER (
        PARTITION BY entity_type, entity_id
        ORDER BY event_time, event_type, transaction_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ), 0) AS previous_known_fraud_count
    FROM linha_do_tempo
),
reportes_por_transacao AS (
    SELECT transaction_id,
        MAX(CASE WHEN entity_type = 'card' THEN previous_known_fraud_count END)
            AS card_previous_known_fraud_count,
        MAX(CASE WHEN entity_type = 'terminal' THEN previous_known_fraud_count END)
            AS terminal_previous_known_fraud_count
    FROM reportes_conhecidos
    WHERE transaction_id IS NOT NULL
    GROUP BY transaction_id
)
SELECT
    historico.* EXCLUDE (card_previous_tx_count, terminal_previous_tx_count),
    CASE WHEN historico.terminal_previous_tx_count = 0 THEN NULL
         ELSE reportes.terminal_previous_known_fraud_count::DOUBLE
              / historico.terminal_previous_tx_count
    END AS terminal_fraud_rate,
    CASE WHEN historico.card_previous_tx_count = 0 THEN NULL
         ELSE reportes.card_previous_known_fraud_count::DOUBLE
              / historico.card_previous_tx_count
    END AS card_fraud_rate
FROM historico_contagens AS historico
JOIN reportes_por_transacao AS reportes USING (transaction_id)
"""

In [51]:
display(db.execute(f"""
    SELECT
        card_id,
        tx_datetime,
        tx_amount,
        card_amount_mean_30d,
        card_amount_std_30d,
        amount_z_score_30d
    FROM ({query_obt}) AS obt
    ORDER BY card_id, tx_datetime
    LIMIT 20
""").df())

,card_id,tx_datetime,tx_amount,card_amount_mean_30d,card_amount_std_30d,amount_z_score_30d
0,60401818053,2018-01-02 08:04:49,30.750000,NaN,NaN,NaN
1,60401818053,2018-01-03 13:02:16,53.360001,30.750000,NaN,NaN
2,60401818053,2018-01-05 11:55:02,110.099998,42.055000,15.987685,4.256088
3,60401818053,2018-01-05 12:00:00,61.090000,64.736666,40.880031,-0.089204
4,60401818053,2018-01-09 06:46:43,78.050003,63.825000,33.428169,0.425539
5,60401818053,2018-01-10 11:57:57,83.930000,66.670000,29.640378,0.582314
6,60401818053,2018-01-10 16:31:37,42.139999,69.546667,27.431604,-0.999091
7,60401818053,2018-01-11 16:58:18,330.850006,65.631429,27.099466,9.786856
8,60401818053,2018-01-12 12:14:43,44.040001,98.783751,97.067405,-0.563977
9,60401818053,2018-01-14 21:31:26,61.009998,92.701112,92.613754,-0.342186


# Validação da OBT

Antes da exportação, conferimos unicidade das transações, indicadores binários, consistência do desvio e do Z-score, intervalos positivos, terminais distintos e presença do alvo.

O total de linhas deve coincidir com o total original e com os identificadores únicos. As contagens de inconsistências devem ser 0. A conferência do esquema deve mostrar exatamente 26 colunas, sem nomes duplicados com sufixos.

`is_fraud` é apenas o alvo. `is_transactional_fraud` e `tx_fraud_report_date` não integram as entradas. Atributos baseados em fraudes reportadas consideram apenas reportes disponíveis antes de cada transação; `is_fraud` continua sendo somente o alvo.


In [52]:
validacao_obt = db.execute(f"""
    WITH obt AS (
        {query_obt}
    )
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT transaction_id) AS ids_unicos,

        COUNT(*) FILTER (
            WHERE terminal_new_for_card NOT IN (0, 1)
               OR terminal_new_for_card IS NULL
        ) AS terminal_novo_invalido,

        COUNT(*) FILTER (
            WHERE card_history_count_30d < 2
              AND card_amount_std_30d IS NOT NULL
        ) AS desvio_inconsistente,

        COUNT(*) FILTER (
            WHERE (
                card_amount_std_30d IS NULL
                OR card_amount_std_30d = 0
            )
            AND amount_z_score_30d IS NOT NULL
        ) AS z_score_inconsistente,

        COUNT(*) FILTER (
            WHERE seconds_since_last_card_tx <= 0
        ) AS intervalos_invalidos,

        COUNT(*) FILTER (
            WHERE card_distinct_terminals_24h < 0
               OR card_distinct_terminals_24h > card_tx_count_24h
               OR card_distinct_terminals_24h IS NULL
        ) AS terminais_distintos_invalidos,

        COUNT(*) FILTER (
            WHERE is_fraud IS NULL
        ) AS alvos_ausentes,

        COUNT(*) FILTER (
            WHERE card_id IS NULL OR terminal_id IS NULL OR tx_datetime IS NULL
        ) AS chaves_ou_tempo_ausentes,

        COUNT(*) FILTER (
            WHERE terminal_tx_count_1h IS NULL
               OR terminal_tx_count_24h IS NULL
               OR terminal_distinct_cards_24h IS NULL
        ) AS contagens_terminal_ausentes,

        COUNT(*) FILTER (
            WHERE terminal_tx_count_1h > terminal_tx_count_24h
               OR terminal_distinct_cards_24h > terminal_tx_count_24h
               OR (
                   seconds_since_last_terminal_tx IS NOT NULL
                   AND seconds_since_last_terminal_tx <= 0
               )
        ) AS historico_terminal_inconsistente,

        COUNT(*) FILTER (
            WHERE card_distinct_terminals < 0
               OR card_distinct_terminals IS NULL
        ) AS terminais_distintos_historicos_invalidos,

        COUNT(*) FILTER (
            WHERE terminal_fraud_rate < 0 OR terminal_fraud_rate > 1
               OR card_fraud_rate < 0 OR card_fraud_rate > 1
        ) AS taxas_fraude_invalidas

    FROM obt
""").df()
display(validacao_obt)

assert validacao_obt.loc[0, "ids_unicos"] == validacao_obt.loc[0, "total_linhas"]
assert validacao_obt.loc[0, "chaves_ou_tempo_ausentes"] == 0
assert validacao_obt.loc[0, "contagens_terminal_ausentes"] == 0
assert validacao_obt.loc[0, "alvos_ausentes"] == 0
assert validacao_obt.loc[0, "terminal_novo_invalido"] == 0
assert validacao_obt.loc[0, "desvio_inconsistente"] == 0
assert validacao_obt.loc[0, "z_score_inconsistente"] == 0
assert validacao_obt.loc[0, "intervalos_invalidos"] == 0
assert validacao_obt.loc[0, "terminais_distintos_invalidos"] == 0
assert validacao_obt.loc[0, "historico_terminal_inconsistente"] == 0
assert validacao_obt.loc[0, "terminais_distintos_historicos_invalidos"] == 0
assert validacao_obt.loc[0, "taxas_fraude_invalidas"] == 0

,total_linhas,ids_unicos,terminal_novo_invalido,desvio_inconsistente,z_score_inconsistente,intervalos_invalidos,terminais_distintos_invalidos,alvos_ausentes,chaves_ou_tempo_ausentes,contagens_terminal_ausentes,historico_terminal_inconsistente,terminais_distintos_historicos_invalidos,taxas_fraude_invalidas
0,4483956,4483956,0,0,0,0,0,0,0,0,0,0,0


In [53]:
schema_obt = db.execute(f"""
    DESCRIBE
    SELECT *
    FROM ({query_obt}) AS obt
""").df()

expected_columns = [
    "transaction_id", "tx_datetime", "tx_amount", "card_id", "terminal_id",
    "terminal_category", "hour", "day_week", "terminal_age_days", "is_fraud",
    "terminal_new_for_card", "card_tx_count_1h", "card_tx_count_24h",
    "card_distinct_terminals_24h", "seconds_since_last_card_tx",
    "terminal_tx_count_1h", "terminal_tx_count_24h",
    "terminal_distinct_cards_24h", "seconds_since_last_terminal_tx",
    "card_history_count_30d", "card_amount_mean_30d", "card_amount_std_30d",
    "amount_z_score_30d",
    "card_distinct_terminals", "terminal_fraud_rate", "card_fraud_rate",
]
assert schema_obt["column_name"].tolist() == expected_columns

display(schema_obt)
print("Quantidade de colunas:", len(schema_obt))

,column_name,column_type,null,key,default,extra
0,transaction_id,BIGINT,YES,None,None,None
1,tx_datetime,TIMESTAMP_NS,YES,None,None,None
2,tx_amount,FLOAT,YES,None,None,None
3,card_id,BIGINT,YES,None,None,None
4,terminal_id,BIGINT,YES,None,None,None
5,terminal_category,VARCHAR,YES,None,None,None
6,hour,BIGINT,YES,None,None,None
7,day_week,BIGINT,YES,None,None,None
8,terminal_age_days,DOUBLE,YES,None,None,None
9,is_fraud,BOOLEAN,YES,None,None,None


Quantidade de colunas: 26


## Exportação da OBT

Salvamos `data/processed/obt.parquet` com compressão ZSTD para reutilizar os atributos sem recalcular todo o histórico. A exportação preserva o banco original e interrompe a execução se o destino já existir.

A consulta calcula os atributos de toda a base nesta etapa e pode demorar. Depois, inspecionamos o esquema do arquivo salvo e a distribuição mensal.


In [54]:
output_path = PROCESSED_DIR / "obt_with_terminal_history.parquet"

if output_path.exists():
    raise FileExistsError(
        f"O arquivo já existe: {output_path}"
    )

output_sql = output_path.as_posix().replace("'", "''")

db.execute(f"""
    COPY (
        {query_obt}
    )
    TO '{output_sql}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"OBT salva em: {output_path}")

OBT salva em: C:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\data\processed\obt_with_terminal_history.parquet


In [55]:
display(db.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{output_sql}')
""").df())

,column_name,column_type,null,key,default,extra
0,transaction_id,BIGINT,YES,None,None,None
1,tx_datetime,TIMESTAMP_NS,YES,None,None,None
2,tx_amount,FLOAT,YES,None,None,None
3,card_id,BIGINT,YES,None,None,None
4,terminal_id,BIGINT,YES,None,None,None
5,terminal_category,VARCHAR,YES,None,None,None
6,hour,BIGINT,YES,None,None,None
7,day_week,BIGINT,YES,None,None,None
8,terminal_age_days,DOUBLE,YES,None,None,None
9,is_fraud,BOOLEAN,YES,None,None,None


In [56]:
display(db.execute(f"""
    SELECT
        DATE_TRUNC('month', tx_datetime) AS mes,
        COUNT(*) AS total_transacoes
    FROM read_parquet('{output_sql}')
    GROUP BY 1
    ORDER BY 1
""").df())

,mes,total_transacoes
0,2018-01-01,765548
1,2018-02-01,767348
2,2018-03-01,929984
3,2018-04-01,966024
4,2018-05-01,1055052


# Divisão temporal em treinamento e validação

O enunciado prevê cinco meses de treinamento e um sexto de validação. A base fornecida contém apenas janeiro a maio de 2018. Para os dados disponíveis, adotamos a divisão autorizada neste projeto:

- Treinamento: janeiro a abril, antes de `2018-05-01` (3.428.904 transações).
- Validação: maio, de `2018-05-01` até antes de `2018-06-01` (1.055.052 transações).

A divisão é cronológica. Os atributos já calculados para maio preservam o histórico anterior, incluindo eventos anteriores do próprio mês, sem consultar o futuro.


In [57]:
query_train = f"""
SELECT *
FROM read_parquet('{output_sql}')
WHERE tx_datetime < TIMESTAMP '2018-05-01 00:00:00'
"""

query_validation = f"""
SELECT *
FROM read_parquet('{output_sql}')
WHERE tx_datetime >= TIMESTAMP '2018-05-01 00:00:00'
  AND tx_datetime < TIMESTAMP '2018-06-01 00:00:00'
"""

In [58]:
display(db.execute(f"""
    SELECT
        'treinamento' AS conjunto,
        COUNT(*) AS total_transacoes,
        MIN(tx_datetime) AS inicio,
        MAX(tx_datetime) AS fim
    FROM ({query_train}) AS treino

    UNION ALL

    SELECT
        'validacao',
        COUNT(*),
        MIN(tx_datetime),
        MAX(tx_datetime)
    FROM ({query_validation}) AS validacao
""").df())

,conjunto,total_transacoes,inicio,fim
0,treinamento,3428904,2018-01-01 00:00:10,2018-04-30 23:59:59
1,validacao,1055052,2018-05-01 00:00:08,2018-05-31 23:59:38


## Exportação dos conjuntos

Salvamos `obt_train.parquet` e `obt_validation.parquet` em `data/processed`, mantendo as 26 colunas para rastreabilidade. Assim como a OBT completa, esses arquivos não são sobrescritos automaticamente.


In [59]:
train_path = PROCESSED_DIR / "obt_train.parquet"
validation_path = PROCESSED_DIR / "obt_validation.parquet"

for arquivo in [train_path, validation_path]:
    if arquivo.exists():
        raise FileExistsError(f"O arquivo já existe: {arquivo}")

train_sql = train_path.as_posix().replace("'", "''")
validation_sql = validation_path.as_posix().replace("'", "''")

db.execute(f"""
    COPY ({query_train})
    TO '{train_sql}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

db.execute(f"""
    COPY ({query_validation})
    TO '{validation_sql}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Treinamento salvo em: {train_path}")
print(f"Validação salva em: {validation_path}")

Treinamento salvo em: C:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\data\processed\obt_train.parquet
Validação salva em: C:\Users\lf446\OneDrive\Documentos\GitHub\ti326m-trabalho3-equipe2\data\processed\obt_validation.parquet


# Dicionário das colunas finais

| Coluna | Finalidade |
|---|---|
| `transaction_id` | Identificador da transação; rastreabilidade. |
| `tx_datetime` | Instante da transação; histórico e divisão temporal. |
| `tx_amount` | Valor da compra. |
| `card_id` | Identificador do cartão; agrupamento do histórico. |
| `terminal_id` | Identificador do terminal. |
| `hour` | Hora da compra (0–23). |
| `day_week` | Dia da semana (0 = domingo, 6 = sábado). |
| `terminal_age_days` | Dias completos de operação do terminal. |
| `is_fraud` | Rótulo-alvo. |
| `terminal_new_for_card` | Primeiro uso observado do terminal pelo cartão. |
| `card_tx_count_1h` | Transações anteriores do cartão em 1h. |
| `card_tx_count_24h` | Transações anteriores do cartão em 24h. |
| `card_distinct_terminals_24h` | Terminais distintos no histórico de 24h. |
| `card_distinct_terminals` | Terminais distintos já usados pelo cartão antes da transação. |
| `terminal_fraud_rate` | Fração de transações anteriores do terminal com fraude já reportada. |
| `card_fraud_rate` | Fração de transações anteriores do cartão com fraude já reportada. |
| `terminal_tx_count_1h` / `terminal_tx_count_24h` | Transações anteriores no terminal. |
| `terminal_distinct_cards_24h` | Cartões distintos no terminal nas 24h anteriores. |
| `seconds_since_last_terminal_tx` | Segundos desde a transação anterior no terminal. |
| `seconds_since_last_card_tx` | Intervalo desde a última transação anterior. |
| `card_history_count_30d` | Valores disponíveis no histórico de 30 dias. |
| `card_amount_mean_30d` | Média histórica dos valores em 30 dias. |
| `card_amount_std_30d` | Desvio padrão amostral histórico em 30 dias. |
| `amount_z_score_30d` | Desvio relativo do valor atual em relação ao histórico. |
| `terminal_category` | Categoria do terminal que a transação ocorreu. | 

Os três identificadores e `tx_datetime` permanecem para conferência, mas ficam fora das entradas iniciais do modelo. `is_fraud` é separado como alvo. Restam 21 atributos candidatos ao treinamento.
